# Local session defaults, plain mode, and dispatch

`dryml.session` is the persistent common path for notebooks. Fresh DRYML intentionally behaves as ordinary unchecked Python. A managed session checks the current process; requested worker environment/world candidates remain separate for later explicit dispatch. This lesson runs offline and creates Store state only in a temporary directory.

Set managed or orchestrator mode before importing project code or TensorFlow, PyTorch, or JAX when visibility guarantees matter. A visibility-changing transition after a known framework import requires a fresh process rather than pretending it was safely undone.

In [ ]:
import operator
import importlib
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import dryml
from dryml.annotations.errors import AnnotationResolutionError
from dryml.core import Definition
from dryml.core.object import Object
from dryml.core.store.dir import DirStore
from dryml.environments import PythonExecutableSpec
from dryml.runtime.allocation import is_no_allocation
from dryml.worlds import LocalResourceInventory


class PlainCounter(Object):
    def __init__(self, value):
        super().__init__()
        self.value = value

    def plus(self, value):
        return self.value + value


## Ordinary Python is the default

Leaving the session untouched, or explicitly calling `set_mode("python")`, preserves normal Python behavior. Its low-level state is `RuntimeMode.NONE + NoAllocation + OFF` with no enabled requirement axes, so there is no session-derived allocation or requirement check.

In [ ]:
python_snapshot = dryml.session.current()
assert dryml.session.mode() == 'python'
assert python_snapshot.allocation is None
assert is_no_allocation(python_snapshot.runtime.allocation)
assert python_snapshot.runtime.mode.value == 'none'
assert python_snapshot.requirement_axes.to_data() == []
assert dryml.session.set_mode('python').mode == 'python'


## Managed current process and immutable snapshots

`manage()` is the short CPU-only checked path. `configure(...)` atomically replaces the complete session when setup belongs in one declarative cell. Process memory is a requirement-compatible declarative allowance, not a hard limit on arbitrary Python allocation.

In [ ]:
assert dryml.session.configure(mode='python').mode == 'python'
managed = dryml.session.manage(cpus=1)
managed = dryml.session.require_env('dryml>=0')
assert managed.mode == 'managed'
assert managed.allocation is not None
assert managed.controls['memory'] in {'undeclared', 'declarative'}
assert managed.statuses['visibility'] == 'visibility-enforced'
try:
    managed.statuses['visibility'] = 'changed'
except TypeError:
    pass
else:
    raise AssertionError('session snapshots must be immutable')

entered = []


@dryml.world.req(cpus={'min': 2})
def needs_two_cpus():
    entered.append('body')


try:
    needs_two_cpus()
except AnnotationResolutionError:
    pass
else:
    raise AssertionError('managed direct requirements must check before the body')
assert entered == []
assert dryml.session.manage(cpus=1).generation == managed.generation


## Worker intent is not the notebook allowance

`worker_env_request(...)` and `worker_world_request(...)` set only concrete default candidates for future workers. `require_env(...)` remains a hard software compatibility requirement. This CPU-only lesson requests one CPU worker. On a GPU host, `worker_world_request(cpus=1, gpus=1)` can dispatch a GPU worker while the managed notebook remains CPU-only; framework imports in the notebook still see the managed visibility policy. Explicit dispatch defaults to strict compatibility on all requirement axes.

In [ ]:
worker_environment = PythonExecutableSpec(sys.executable, pythonpath_policy='none')
requested = dryml.session.worker_env_request(worker_environment)
requested = dryml.session.worker_world_request(cpus=1)
assert requested.allocation == managed.allocation
assert requested.requested_environment is not None
assert requested.requested_world is not None
inventory = LocalResourceInventory((0,))

with TemporaryDirectory(prefix='dryml-session-tutorial-') as directory:
    store = DirStore(directory, query_index='none')
    try:
        explanation = dryml.dispatch.explain(
            operator.add, store=store, inventory=inventory, args=(20, 22)
        )
        summary = {
            'launchable': explanation.launchable,
            'environment_source': explanation.resolution.environment_selection.source,
            'world_source': explanation.resolution.world_selection.source,
            'diagnostic_count': len(explanation.resolution.diagnostics),
        }
        assert summary['launchable'] is True
        assert summary['environment_source'] == 'session_requested'
        assert summary['world_source'] == 'session_requested'
        result = dryml.dispatch.run(
            operator.add,
            store=store,
            inventory=inventory,
            args=(20, 22),
        )
        assert result.status == 'ok'
        assert result.result_canonical == 42
    finally:
        store.close()


## Advanced low-level inline work

`runtime.plain()` remains available for advanced trusted inline work. It is not a managed session and not worker isolation. Keep this scoped API for cases that need its exact lifetime semantics.

In [ ]:
with dryml.runtime.plain() as inline_runtime:
    assert inline_runtime is dryml.runtime.active_runtime()
    counter = Definition(PlainCounter, 3).build()
    assert counter.plus(4) == 7

reset = dryml.session.reset()
assert reset.mode == 'python'
assert reset.allocation is None
assert reset.requested_world is None
assert reset.environment.requirements == ()


## Strict orchestration is definition-only

After setup, strict orchestration is session-wide definition mode: definition, concrete, selector, and space work remain available, but `fresh`, `load_or_build`, and local managed execution are blocked. Object materialization raises `Orchestration mode prohibits Object materialization`; dispatch to a worker for live workload execution. The parent remains accelerator-free while a future worker request may select accelerators.

## Disposable fake-framework restart boundary

The final cell creates an offline fake TensorFlow root to demonstrate that a raw registered import sees CPU-only visibility and finalizes its status before returning. The runner executes this notebook in a disposable child, so the handled restart-required state is valid only there. Do not use this pattern to continue a reusable notebook after a visibility-changing framework import.

In [ ]:
with TemporaryDirectory(prefix='dryml-session-fake-framework-') as directory:
    package = Path(directory) / 'tensorflow'
    package.mkdir()
    (package / '__init__.py').write_text(
        "import os\n"
        "SEEN = os.environ.get('CUDA_VISIBLE_DEVICES')\n"
        "class _Config:\n"
        "    @staticmethod\n"
        "    def get_physical_devices(kind): return ()\n"
        "    @staticmethod\n"
        "    def set_visible_devices(devices, kind): pass\n"
        "    @staticmethod\n"
        "    def get_visible_devices(kind): return ()\n"
        "config = _Config()\n",
        encoding='utf-8',
    )
    sys.path.insert(0, directory)
    try:
        pending = dryml.session.set_mode('orchestrator')
        assert dryml.status()['object_mode'] == 'definition'
        assert pending.statuses['tensorflow:tensorflow:visibility'] == 'pending-import'
        tensorflow = importlib.import_module('tensorflow')
        assert tensorflow.SEEN == ''
        assert dryml.session.current().statuses['tensorflow:tensorflow:visibility'] == 'visibility-enforced'
        try:
            dryml.session.reset()
        except RuntimeError as error:
            assert 'restart' in str(error)
        else:
            raise AssertionError('a visibility-changing reset after framework import requires restart')
    finally:
        sys.path.remove(directory)

NOTEBOOK_RESTART_REQUIRED_HANDLED = True


For raw TensorFlow, PyTorch, or JAX imports, establish managed/orchestrator setup first. Registered hooks make mandatory visibility fail closed and report optional controls per adapter. If a framework already imported and a visibility change is needed, use a disposable fresh process rather than attempting a reusable-notebook reset. See `docs/session.md` and `docs/migration/session_runtime_default.md`.